## Running Yeast9 model with all different kcats

Here I will construct the different ecGEMs and run FBA for every of them

For further details on how to install COBRA-k, please refer to the original documentation: https://klamt-lab.github.io/COBRA-k/index.html

**Requirements**

COBRA-k requires the specification of the solver path before loading the package.

In [1]:
import os

scip_path =  "your_scip_path"

os.environ["PATH"] = scip_path + ":" + os.environ["PATH"]

In [2]:
# Necessary packages
import cobrak
from cobrak.io import load_annotated_sbml_model_as_cobrak_model
import json
from cobrak.lps import perform_lp_optimization
import pandas as pd
import copy
from cobrak.thermokinetic_data_retrieval import add_thermokinetic_data_to_cobrak_model
import glob
import numpy as np

In [3]:
path_to_sbml = "../model/yeast-GEM.xml"


In [4]:
model = load_annotated_sbml_model_as_cobrak_model(
    path_to_sbml,
    do_model_fullsplit=True
)

INFO: No molecular weight given as cobrak_mw annotation for Q0045. Setting to standard value 1000000.0.
 Please change this value later to a reasonable value if you use enzyme constraints, e.g. through COBRA-k's Uniprot functionality.
INFO: No molecular weight given as cobrak_mw annotation for Q0080. Setting to standard value 1000000.0.
 Please change this value later to a reasonable value if you use enzyme constraints, e.g. through COBRA-k's Uniprot functionality.
INFO: No molecular weight given as cobrak_mw annotation for Q0085. Setting to standard value 1000000.0.
 Please change this value later to a reasonable value if you use enzyme constraints, e.g. through COBRA-k's Uniprot functionality.
INFO: No molecular weight given as cobrak_mw annotation for Q0105. Setting to standard value 1000000.0.
 Please change this value later to a reasonable value if you use enzyme constraints, e.g. through COBRA-k's Uniprot functionality.
INFO: No molecular weight given as cobrak_mw annotation for 

Since Yeast9 has its glucose uptake constrained by default, we will open it in the next step to make sure that there will be no constrain other than the kcats

In [5]:
model_2 = copy.deepcopy(model)
model_2.reactions["r_1714_REV"].max_flux=1000

In [7]:
# kcat dicts
tools_path = f'../data/yeast/*_yeast_kcat_dict.json'
catpsn = glob.glob(tools_path, recursive=True)


#kcat_dict.sort()
print(catpsn)
len(catpsn)

['../datap/yeast/brenda_yeast_kcat_dict.json', '../datap/yeast/catpred_yeast_kcat_dict.json', '../datap/yeast/dlkcat_yeast_kcat_dict.json', '../datap/yeast/mean_yeast_kcat_dict.json', '../datap/yeast/catapro_yeast_kcat_dict.json', '../datap/yeast/mmkcat_yeast_kcat_dict.json', '../datap/yeast/turnup_yeast_kcat_dict.json', '../datap/yeast/unikp_yeast_kcat_dict.json']


8

In [9]:
## MW
# Molecular Weight dictionary
with open(f'../data/json_mw_dict.json', 'r') as file:
    mw_dict = json.load(file)

## Growth predictions

Here I will:
1. Convert the values of the kcats dictionaires to $h^{-1}$ instead $s^{-1}$
2. Add the thermokinetic data: kcats + molecular weights
3. Perform FBA predictions with the enzime constraints
4. Save growth value

### Tools, enviromental and mean interval growth

In [10]:
results = {}

for i, catpn in enumerate(catpsn):
    print(f'Loading tool {i+1:02d}/{len(catpsn)}', catpn)
    
    model_k = copy.deepcopy(model_2)
    
    with open(catpn, 'r') as file:
        kcat_dict = json.load(file)

    kcat_dict_h = {k: v * 3600 for k, v in kcat_dict.items()}

    add_thermokinetic_data_to_cobrak_model(
        cobrak_model=model_k,
        kcats=kcat_dict_h,
        mws=mw_dict
    )

    fba_results = perform_lp_optimization(
        model_k,
        "r_2111",
        +1,
        with_enzyme_constraints=True
    )

    growth = fba_results["r_2111"]
    print(catpn, "Growth:", growth)

    results[catpn] = growth  

Loading tool 01/8 ../datap/yeast/brenda_yeast_kcat_dict.json
../datap/yeast/brenda_yeast_kcat_dict.json Growth: 0.3404577099936432
Loading tool 02/8 ../datap/yeast/catpred_yeast_kcat_dict.json
../datap/yeast/catpred_yeast_kcat_dict.json Growth: 0.0
Loading tool 03/8 ../datap/yeast/dlkcat_yeast_kcat_dict.json
../datap/yeast/dlkcat_yeast_kcat_dict.json Growth: 0.3940887488949131
Loading tool 04/8 ../datap/yeast/mean_yeast_kcat_dict.json
../datap/yeast/mean_yeast_kcat_dict.json Growth: 2.7478919856382262
Loading tool 05/8 ../datap/yeast/catapro_yeast_kcat_dict.json
../datap/yeast/catapro_yeast_kcat_dict.json Growth: 0.21473216530284844
Loading tool 06/8 ../datap/yeast/mmkcat_yeast_kcat_dict.json
../datap/yeast/mmkcat_yeast_kcat_dict.json Growth: 0.30321084978405705
Loading tool 07/8 ../datap/yeast/turnup_yeast_kcat_dict.json
../datap/yeast/turnup_yeast_kcat_dict.json Growth: 0.6167244050884619
Loading tool 08/8 ../datap/yeast/unikp_yeast_kcat_dict.json
../datap/yeast/unikp_yeast_kcat_dict

### Random dictionaries growth iteration

From all_kcats list we will get the max and min kcat value to use as the interval and n_reactions wil be the total number of reactions inside of Yeats9 model

In [23]:
all_kcats=pd.read_csv('../data/all_kcats.csv').iloc[:, 0].tolist()

In [25]:
log_kcats = np.log10(all_kcats)

log_min = log_kcats.min()
log_max = log_kcats.max()

mu = log_kcats.mean()
sigma = log_kcats.std()

print("log_min:", log_min)
print("log_max:", log_max)

log_min: -3.3979400086720375
log_max: 7.641959507450011


In [36]:
n_reactions = len(np.load("../data/reactions.npy", allow_pickle=True))

In [40]:
n_iter = 100
# store all generated kcats
uniform_kcat_runs = []
gauss_kcat_runs = []
# save growth
growth_uniform = []
growth_gauss = []

for i in range(n_iter):

    print(f"Iteration {i+1}/{n_iter}")

    
    # 1. UNIFORM RANDOMIZATION
   
    random_log_uniform = np.random.uniform(
        log_min,
        log_max,
        n_reactions
    )

    random_kcat_uniform = 10 ** random_log_uniform

    # save generated kcats
    uniform_kcat_runs.append(random_kcat_uniform)

    # convert to dictionary
    kcat_uniform_dict = dict(
        zip(reaction_ids, random_kcat_uniform)
    )

    # convert to hours
    kcat_uniform_dict_h = {
        k: v * 3600
        for k, v in kcat_uniform_dict.items()
    }

    # copy model
    model_uniform = copy.deepcopy(model)

    model_uniform.reactions["r_1714_REV"].max_flux = 1000

    # add kcats
    add_thermokinetic_data_to_cobrak_model(
        cobrak_model=model_uniform,
        kcats=kcat_uniform_dict_h,
        mws=mw_dict
    )

    try:

        fba_results = perform_lp_optimization(
            model_uniform,
            {"r_2111": +1},
            +1,
            with_enzyme_constraints=True
        )

        growth = fba_results["r_2111"]

        if growth is None or np.isnan(growth):

            growth_uniform.append(np.nan)

        else:

            growth_uniform.append(growth)

    except Exception:

        growth_uniform.append(np.nan)

   
    # 2. GAUSSIAN RANDOMIZATION
    
    random_log_gauss = np.random.normal(
        mu,
        sigma,
        n_reactions
    )


    random_kcat_gauss = 10 ** random_log_gauss

    # save generated kcats
    gauss_kcat_runs.append(random_kcat_gauss)

    # dictionary
    kcat_gauss_dict = dict(
        zip(reaction_ids, random_kcat_gauss)
    )

    # convert to hours
    kcat_gauss_dict_h = {
        k: v * 3600
        for k, v in kcat_gauss_dict.items()
    }

    # copy model
    model_gauss = copy.deepcopy(model)

    model_gauss.reactions["r_1714_REV"].max_flux = 1000

    # add kcats
    add_thermokinetic_data_to_cobrak_model(
        cobrak_model=model_gauss,
        kcats=kcat_gauss_dict_h,
        mws=mw_dict
    )

    try:

        fba_results = perform_lp_optimization(
            model_gauss,
            {"r_2111": +1},
            +1,
            with_enzyme_constraints=True
        )

        growth = fba_results["r_2111"]

        if growth is None or np.isnan(growth):

            growth_gauss.append(np.nan)

        else:

            growth_gauss.append(growth)

    except Exception:

        growth_gauss.append(np.nan)


# SAVE GROWTH RESULTS

df_results = pd.DataFrame({
    "uniform_growth": growth_uniform,
    "gauss_growth": growth_gauss
})



Iteration 1/100
Iteration 2/100
Iteration 3/100
Iteration 4/100
Iteration 5/100
Iteration 6/100
Iteration 7/100
Iteration 8/100
Iteration 9/100
Iteration 10/100
Iteration 11/100
Iteration 12/100
Iteration 13/100
Iteration 14/100
Iteration 15/100
Iteration 16/100
Iteration 17/100
Iteration 18/100
Iteration 19/100
Iteration 20/100
Iteration 21/100
Iteration 22/100
Iteration 23/100
Iteration 24/100
Iteration 25/100
Iteration 26/100
model.name="unknown";
    - termination condition: infeasible
    - message from solver: infeasible
Iteration 27/100
Iteration 28/100
Iteration 29/100
Iteration 30/100
Iteration 31/100
Iteration 32/100
Iteration 33/100
Iteration 34/100
Iteration 35/100
Iteration 36/100
Iteration 37/100
Iteration 38/100
Iteration 39/100
model.name="unknown";
    - termination condition: infeasible
    - message from solver: infeasible
Iteration 40/100
Iteration 41/100
Iteration 42/100
Iteration 43/100
Iteration 44/100
Iteration 45/100
Iteration 46/100
Iteration 47/100
model.name

In [43]:
df_results.describe()

,uniform_growth,gauss_growth
count,9.100000e+01,100.000000
mean,-3.582087e-12,0.126985
std,1.247812e-11,0.051159
min,-5.820766e-11,0.016614
25%,0.000000e+00,0.099984
50%,0.000000e+00,0.127437
75%,0.000000e+00,0.150002
max,2.072804e-11,0.386359


### Enviromental growth

In [44]:
model_k = copy.deepcopy(model_2)

In [45]:
r = model_k.reactions["r_1714_REV"]
r.min_flux, r.max_flux = 3.9, 3.9

In [46]:
fba_results_exp = perform_lp_optimization(
        model_k,
        "r_2111",
        +1,
        with_enzyme_constraints=False
    )

print(fba_results_exp["r_2111"])

0.34421505266800523


### CatPred ensemble dictionaries growth

In [47]:
catp_path = f'../data/yeast/catpred_kcat_max_model_*.json'

In [49]:
catpsn = glob.glob(catp_path, recursive=True)

catpsn.sort()
print(catpsn[0])
len(catpsn)

../datap/yeast/catpred_kcat_max_model_0.json


10

In [ ]:
results = {}

for i, catpn in enumerate(catpsn):
    print(f'Loading tool {i+1:02d}/{len(catpsn)}', catpn)
    
    model_k = copy.deepcopy(model_2)
    
    with open(catpn, 'r') as file:
        kcat_dict = json.load(file)

    kcat_dict_h = {k: v * 3600 for k, v in kcat_dict.items()}

    add_thermokinetic_data_to_cobrak_model(
        cobrak_model=model_k,
        kcats=kcat_dict_h,
        mws=mw_dict
    )

    fba_results = perform_lp_optimization(
        model_k,
        "r_2111",
        +1,
        with_enzyme_constraints=True
    )

    growth = fba_results["r_2111"]
    print(catpn, "Growth:", growth)

    results[catpn] = growth

Loading tool 01/10 ../datap/yeast/catpred_kcat_max_model_0.json
../datap/yeast/catpred_kcat_max_model_0.json Growth: 0.2554890291939955
Loading tool 02/10 ../datap/yeast/catpred_kcat_max_model_1.json
../datap/yeast/catpred_kcat_max_model_1.json Growth: 6.21841286609033e-11
Loading tool 03/10 ../datap/yeast/catpred_kcat_max_model_2.json
../datap/yeast/catpred_kcat_max_model_2.json Growth: 0.2987142368359041
Loading tool 04/10 ../datap/yeast/catpred_kcat_max_model_3.json
../datap/yeast/catpred_kcat_max_model_3.json Growth: 2.9103830456733704e-11
Loading tool 05/10 ../datap/yeast/catpred_kcat_max_model_4.json
../datap/yeast/catpred_kcat_max_model_4.json Growth: 0.25574180370313115
Loading tool 06/10 ../datap/yeast/catpred_kcat_max_model_5.json
../datap/yeast/catpred_kcat_max_model_5.json Growth: 0.23586350098091594
Loading tool 07/10 ../datap/yeast/catpred_kcat_max_model_6.json
../datap/yeast/catpred_kcat_max_model_6.json Growth: 0.17035066727875567
Loading tool 08/10 ../datap/yeast/catpr